# BEM surface (NOT READY)

The default approach for building BEMs (Boundary Element Models) in MNE involves generating the skull and skin surfaces using FreeSurfer’s watershed algorithm. This algorithm accurately computes the brain and skin surfaces, then expands or contracts these surfaces to estimate the skull. However, this often results in suboptimal surface quality.

An alternative algorithm, mri_bem_surfaces, can produce better results but requires FLASH images, which are not always available.

In this workflow, we use a different approach. Using the CHARM pipeline implemented in SimNIBS, we first generate a segmentation of the various tissue types. We then extract these tissues to create binary masks, which are converted into surfaces using FreeSurfer. The final step involves correcting the surfaces to ensure that they all share the same spherical topology.

Prerequisits:
- CHARM software
- FreeSurfer software

Steps
1. Generate head segmentation
2. Extract binary masks for surface generationG
3. Generate surfaces
4. Correct surfaces

**We will generate head segmentations using the [CHARM](https://www.sciencedirect.com/science/article/pii/S1053811920305309?via%3Dihub) pipeline implemented through the [SimNIBS](https://simnibs.github.io/simnibs/build/html/index.html) software. Please install SimNIBS following their [instructions](https://simnibs.github.io/simnibs/build/html/installation/simnibs_installer.html) before continuing.** <br>

To use Charm in a terminal, head to the folder containing the T1 (and T2 optional) and run the following command line: 
```bash
charm subID path/to/T1.nii.gz path/to/T2.nii.gz --segment --initatlas
```

The subject ID (subID) is given as first argument. Charm will create a folder named `m2m_subID` that contains the segmentation results. The input images are given as final arguments (first the T1, then the T2, where T2 is optional).

Alternatively, run the following cells which will run the command line for the whole dataset.

📋 **Requirements**:
- FreeSurfer reconstruction.


## 📦 Imports

In [1]:
import mindmap as mm

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- FREESURFER reconstruction folder (`FS_DIR`):
    - T1

📤 **<font color=pink>Outputs</font>**<br>
At `[FS_DIR]/[subID]/mri` you will find a new folder `m2m_[subID]/segmentation` containing:
- `labeling.nii.gz`: Head segmentation volume.
- `labeling_LUT.txt`: Segementation's color LUT.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.

In [2]:
# ------------ Parameters -------------
fconfig    = "../../papers/spike/configuration.yml"
subject    = "pat11"
session    = "preop"
resolution = "cmp"
suffix     = "T1w"
extension  = "nii.gz"
# -------------------------------------

config = mm.io.read_config(fconfig)

t1_file = mm.io.get_bidspath(
    root       = config["CMP_DIR"],
    subject    = subject,
    datatype   = "anat",
    session    = session,
    resolution = resolution,
    suffix     = suffix,
    extension  = extension,
    mne        = False,
)

print(t1_file)

/media/nino/DATA/SPIKE/derivatives/cmp-v3.0.0-beta-RC1/sub-pat11/ses-preop/anat/sub-pat11_ses-preop_res-cmp_T1w.nii.gz


## ⚙️ 1. Generate head segmentation

In [12]:
_ = mm.forward.run_charm(subject, t1_file, overwrite=True)

📊 **Segmentation**

In [3]:
seg = mm.io.get_charmpath(t1_file.parent, subject, "seg")
lut = mm.io.get_charmpath(t1_file.parent, subject, "lut")

mm.viz.plot_segmentation(t1_file, seg, lut)

```
mri_tessellate $SUBJECTS_DIR/subject_name/mri/inner_skull_mask.nii.gz 1 inner_skull_temp
mris_convert $SUBJECTS_DIR/subject_name/surf/inner_skull_temp.orig $SUBJECTS_DIR/subject_name/bem/inner_skull.surf

mri_tessellate $SUBJECTS_DIR/subject_name/mri/outer_skull_mask.nii.gz 1 outer_skull_temp
mris_convert $SUBJECTS_DIR/subject_name/surf/outer_skull_temp.orig $SUBJECTS_DIR/subject_name/bem/outer_skull.surf

mri_tessellate $SUBJECTS_DIR/subject_name/mri/scalp_mask.nii.gz 1 outer_skin_temp
mris_convert $SUBJECTS_DIR/subject_name/surf/outer_skin_temp.orig $SUBJECTS_DIR/subject_name/bem/outer_skin.surf
```

In [ ]:
import mne
import os

# Set up paths (assuming your SIMNIBS masks are in a NIfTI format)
subjects_dir = os.environ['SUBJECTS_DIR']
subject = 'subject_name'

# 1. Use SIMNIBS/SIMNIBS-derived masks to generate the surfaces
# (as demonstrated with mri_tessellate in Step 1)

# 2. Use MNE-Python to read, fix, and prepare the surfaces
# MNE's `make_bem_model` or similar functions will read these surfaces, 
# perform necessary checks (like ensuring no intersections between layers), 
# and may also do some light filtering/smoothing.

# Load the surfaces created from mri_tessellate:
surfaces = mne.make_bem_model(
    subject=subject, 
    ico=4, # Use a lower resolution to reduce file size (adjust as needed)
    conductivity=[0.3, 0.006, 0.3], # Example conductivity values (CSF, Skull, Scalp)
    subjects_dir=subjects_dir,
    # The key here is that MNE will look for the .surf files in the 'bem' directory 
    # and perform the necessary checks for BEM solution computation.
)

# You would then typically compute the BEM solution:
bem = mne.make_bem_solution(surfaces) 
# ... and continue with source space setup.